In [2]:
import pandas as pd

In [3]:
patients = pd.read_csv('patients.csv')
treatments = pd.read_csv('treatments.csv')
adverse_reactions = pd.read_csv('adverse_reactions.csv')
treatments_cut = pd.read_csv('treatments_cut.csv')

In [5]:
 patients.head(2)

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
0,1,female,Zoe,Wellish,576 Brown Bear Drive,Rancho California,California,92390.0,United States,951-719-9170ZoeWellish@superrito.com,7/10/1976,121.7,66,19.6
1,2,female,Pamela,Hill,2370 University Hill Road,Armstrong,Illinois,61812.0,United States,PamelaSHill@cuvox.de+1 (217) 569-3204,4/3/1967,118.8,66,19.2


In [7]:
# export data for manual assessment

with pd.ExcelWriter('clinical_trials.xlsx') as writer:
  patients.to_excel(writer,sheet_name='patients')
  treatments.to_excel(writer,sheet_name='treatments')
  treatments_cut.to_excel(writer,sheet_name='treatment_cut')
  adverse_reactions.to_excel(writer,sheet_name='adverse_reactions')

In [8]:
 patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    503 non-null    int64  
 1   assigned_sex  503 non-null    object 
 2   given_name    503 non-null    object 
 3   surname       503 non-null    object 
 4   address       491 non-null    object 
 5   city          491 non-null    object 
 6   state         491 non-null    object 
 7   zip_code      491 non-null    float64
 8   country       491 non-null    object 
 9   contact       491 non-null    object 
 10  birthdate     503 non-null    object 
 11  weight        503 non-null    float64
 12  height        503 non-null    int64  
 13  bmi           503 non-null    float64
dtypes: float64(3), int64(2), object(9)
memory usage: 55.1+ KB


,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
471,472,female,Nadwah,Naifeh,69 Mattson Street,Portland,OR,97205.0,United States,503-417-1995NadwahHawadahNaifeh@einrot.com,3/12/1957,192.7,66,31.1
433,434,male,Leo,Vieira,4894 Chandler Hollow Road,Pittsburgh,PA,15215.0,United States,412-449-0733LeoDVieira@cuvox.de,3/9/1975,154.4,71,21.5


In [14]:
len(patients[patients['address'].isnull()])

12

In [16]:
patients.duplicated().sum()

np.int64(0)

In [18]:
len(patients[patients.duplicated(subset=["given_name","surname"])])

5

In [19]:
patients.describe()

,patient_id,zip_code,weight,height,bmi
count,503.000000,491.000000,503.000000,503.000000,503.000000
mean,252.000000,49084.118126,173.434990,66.634195,27.483897
std,145.347859,30265.807442,33.916741,4.411297,5.276438
min,1.000000,1002.000000,48.800000,27.000000,17.100000
25%,126.500000,21920.500000,149.300000,63.000000,23.300000
50%,252.000000,48057.000000,175.300000,67.000000,27.200000
75%,377.500000,75679.000000,199.500000,70.000000,31.750000
max,503.000000,99701.000000,255.900000,79.000000,37.700000


In [88]:
# making copy
patients_df = patients.copy()
treatments_df = treatments.copy()
treatments_cut_df = treatments_cut.copy()
adverse_reactions_df = adverse_reactions.copy()

In [89]:
# Create a copy with the new values
patients_df = patients_df.fillna('No data available').infer_objects()

In [90]:
treatments_df['hba1c_change'] = treatments_df['hba1c_start'] - treatments_df['hba1c_end']
treatments_cut_df['hba1c_change'] = treatments_cut_df['hba1c_start'] - treatments_cut_df['hba1c_end']

In [91]:
# 1. Extract the Phone Number
# This regex looks for patterns with dashes or parentheses and international codes
phone_regex = r'(\+?\d?[\s\(\-]?\d{3}[\s\)\-]?\s?\d{3}[\-\s]?\d{4})'

# 2. Extract the Email
# This regex looks for the standard alphanumeric@domain.extension pattern
email_regex = r'([a-zA-Z][a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})'

# # Apply the extraction
patients_df['phone_number'] = patients_df['contact'].str.extract(phone_regex)
patients_df['email'] = patients_df['contact'].str.extract(email_regex)

# drop the original column
patients_df.drop(columns=['contact'], inplace=True)

In [92]:
treatments_df=pd.concat([treatments_df, treatments_cut_df],ignore_index=True)

In [93]:
treatments_df = treatments_df.melt(id_vars=['given_name', 'surname', 'hba1c_start',
       'hba1c_end', 'hba1c_change'],var_name='type', value_name='dosage')

In [94]:
treatments_df = treatments_df[treatments_df["dosage"]!='-'].reset_index(drop=True)

In [95]:
treatments_df

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,type,dosage
0,veronika,jindrová,7.63,7.20,0.43,auralin,41u - 48u
1,skye,gormanston,7.97,7.62,0.35,auralin,33u - 36u
2,sophia,haugen,7.65,7.27,0.38,auralin,37u - 42u
3,eddie,archer,7.89,7.55,0.34,auralin,31u - 38u
4,asia,woźniak,7.76,7.37,0.39,auralin,30u - 36u
...,...,...,...,...,...,...,...
345,christopher,woodward,7.51,7.06,0.45,novodra,55u - 51u
346,maret,sultygov,7.67,7.30,0.37,novodra,26u - 23u
347,lixue,hsueh,9.21,8.80,0.41,novodra,22u - 23u
348,jakob,jakobsen,7.96,7.51,0.45,novodra,28u - 26u


In [96]:
regex_pattern = r'(\d+)[^0-9\-]*\-[^0-9\-]*(\d+)'

# Apply extraction
temp_df = treatments_df['dosage'].str.extract(regex_pattern)
treatments_df["dosage_start"] = temp_df[0]
treatments_df["dosage_end"] = temp_df[1]
treatments_df.head()

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,type,dosage,dosage_start,dosage_end
0,veronika,jindrová,7.63,7.20,0.43,auralin,41u - 48u,41,48
1,skye,gormanston,7.97,7.62,0.35,auralin,33u - 36u,33,36
2,sophia,haugen,7.65,7.27,0.38,auralin,37u - 42u,37,42
3,eddie,archer,7.89,7.55,0.34,auralin,31u - 38u,31,38
4,asia,woźniak,7.76,7.37,0.39,auralin,30u - 36u,30,36


In [97]:
treatments_df.drop(columns=['dosage'], inplace=True)
treatments_df.head()

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,type,dosage_start,dosage_end
0,veronika,jindrová,7.63,7.20,0.43,auralin,41,48
1,skye,gormanston,7.97,7.62,0.35,auralin,33,36
2,sophia,haugen,7.65,7.27,0.38,auralin,37,42
3,eddie,archer,7.89,7.55,0.34,auralin,31,38
4,asia,woźniak,7.76,7.37,0.39,auralin,30,36


In [99]:
treatments_df.merge(adverse_reactions_df,how="left",on=['given_name','surname'])

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,type,dosage_start,dosage_end,adverse_reaction
0,veronika,jindrová,7.63,7.20,0.43,auralin,41,48,NaN
1,skye,gormanston,7.97,7.62,0.35,auralin,33,36,NaN
2,sophia,haugen,7.65,7.27,0.38,auralin,37,42,NaN
3,eddie,archer,7.89,7.55,0.34,auralin,31,38,NaN
4,asia,woźniak,7.76,7.37,0.39,auralin,30,36,NaN
...,...,...,...,...,...,...,...,...,...
345,christopher,woodward,7.51,7.06,0.45,novodra,55,51,nausea
346,maret,sultygov,7.67,7.30,0.37,novodra,26,23,NaN
347,lixue,hsueh,9.21,8.80,0.41,novodra,22,23,injection site discomfort
348,jakob,jakobsen,7.96,7.51,0.45,novodra,28,26,hypoglycemia
